# SKANN-SSL Event Normalisation — 50 Windows Per Event

**Project:** SKANN-SSL — Passive Acoustic Vessel Detection & Classification  
**Deployment:** NUWR Goa, seabed hydrophone, 30 m depth, 512 Hz, 28 Jan – 11 Feb 2026  
**Prepared by:** Oravont Systems LLP for Altair Infrasec Pvt Ltd

---

## Purpose

Normalise training exposure across all 141 vessel events by producing exactly 50 windows
per event into a single flat output folder. Without this, a 342-window event receives
21× more gradient updates than a 16-window event during Barlow Twins training.

## Design

**Short events (< 50 windows)**
- All original windows copied first
- Remaining slots filled by augmenting randomly sampled originals (with replacement)
- Augmentation pipeline (applied independently per window):
  1. Phase shift: `np.roll(x, U(-51, 51))` — simulates bearing/range variation
  2. Variance scale: `x * (1 + N(0, 0.10))` — simulates gain drift
  3. Ambient inject: `x + alpha * Pool_Q_sample` (α ~ U(0.1, 0.5))
     or `x + alpha * Pool_T_sample` (α ~ U(0.3, 0.8), p=0.20)
  4. Re-normalise: z-score
- Augmented files saved as `v_{event_id}_{offset}_aug{n}.npy`

**Long events (> 50 windows)**
- Most diverse 50 selected by greedy max-coverage in IQR-scaled 6D feature space
- Greedy: seed with window furthest from centroid, iteratively add window
  maximising minimum cosine distance to already-selected set
- No augmentation — originals copied as-is

**Exactly 50 windows**
- Originals copied as-is

## Output

| Item | Path |
|---|---|
| Normalised tensors | `SKANN_SSL/ssl_data_50w/v_*.npy` (7,050 files) |
| Pool manifest | `SKANN_SSL/ssl_data_50w/window_pool_50.csv` |

## window_pool_50.csv schema

`tensor_file, event_id, offset_sec, source, split, pre_norm_rms, peakiness,
logbp_1_5, logbp_5_15, logbp_15_40, logbp_40_90, logbp_90_180`

- `source`: `original` or `augmented`
- `split`: `train` or `val` — re-derived from session dates (val = 7, 8, 9 Feb 2026)
- Feature columns: copied from originals; set to 0.0 for augmented (recomputed at training)

## Next step

Feed `window_pool_50.csv` into `pairing_manifest_generator_v8.ipynb` as
`VESSEL_FEATURES_PATH`. Update `TENSOR_BASE_DIR` to `ssl_data_50w/`.


## Cell 1 — Imports + Mount Drive

In [ ]:
import os
import ast
import numpy as np
import pandas as pd
from google.colab import drive

drive.mount('/content/drive')
print('Drive mounted.')

Mounted at /content/drive
Drive mounted.


## Cell 2 — Configuration

In [ ]:
# ── Input paths ────────────────────────────────────────────────────────────
VESSEL_FEATURES_PATH  = '/content/drive/MyDrive/SKANN_SSL/ssl_data_30s/manifests/vessel_window_features.csv'
VESSEL_MANIFEST_PATH  = '/content/drive/MyDrive/SKANN_SSL/ssl_data_30s/manifests/vessel_manifest.csv'
TENSOR_BASE_DIR      = '/content/drive/MyDrive/SKANN_SSL/ssl_data_30s/tensors/vessel'
# Ambient pool tensor directories (for ambient injection during augmentation)
AMBIENT_Q_TENSOR_DIR = '/content/drive/MyDrive/SKANN_SSL/ssl_data_30s/tensors/ambient_Q'
AMBIENT_T_TENSOR_DIR = '/content/drive/MyDrive/SKANN_SSL/ssl_data_30s/tensors/ambient_T'
POOL_Q_MANIFEST_PATH = '/content/drive/MyDrive/SKANN_SSL/ssl_data_30s/manifests/pool_q_manifest.csv'
POOL_T_MANIFEST_PATH = '/content/drive/MyDrive/SKANN_SSL/ssl_data_30s/manifests/pool_t_manifest.csv'

# ── Output paths ────────────────────────────────────────────────────────────
# Tensors written here — same folder as existing 4,394 originals already copied
OUT_TENSOR_DIR    = '/content/drive/MyDrive/SKANN_SSL/v5_data/tensors/vessel'
# Manifests written here
OUT_MANIFEST_DIR  = '/content/drive/MyDrive/SKANN_SSL/ssl_data_50w'
OUT_MANIFEST_PATH = '/content/drive/MyDrive/SKANN_SSL/ssl_data_50w/window_pool_50.csv'

# ── Normalisation parameters ────────────────────────────────────────────────
TARGET_WINDOWS   = 50       # windows per event
PHASE_SHIFT_MAX  = 51       # ± samples at 512 Hz (~± 100 ms)
VAR_SIGMA        = 0.10     # variance scaling std dev
ALPHA_Q_LO       = 0.10     # Pool Q ambient injection lower bound
ALPHA_Q_HI       = 0.50     # Pool Q ambient injection upper bound
ALPHA_T_LO       = 0.30     # Pool T ambient injection lower bound
ALPHA_T_HI       = 0.80     # Pool T ambient injection upper bound
POOL_T_PROB      = 0.20     # probability of using Pool T vs Pool Q
RANDOM_SEED      = 42

# ── Feature columns ─────────────────────────────────────────────────────────
FEAT_COLS = ['logbp_1_5', 'logbp_5_15', 'logbp_15_40',
             'logbp_40_90', 'logbp_90_180', 'peakiness']

os.makedirs(OUT_TENSOR_DIR,   exist_ok=True)
os.makedirs(OUT_MANIFEST_DIR, exist_ok=True)

print('Configuration:')
print(f'  Source tensors           : {TENSOR_BASE_DIR}')
print(f'  Output tensors           : {OUT_TENSOR_DIR}')
print(f'  Output manifest          : {OUT_MANIFEST_PATH}')
print(f'  Target windows per event : {TARGET_WINDOWS}')
print(f'  Phase shift max          : ±{PHASE_SHIFT_MAX} samples')
print(f'  Variance sigma           : {VAR_SIGMA}')
print(f'  Pool Q alpha             : {ALPHA_Q_LO}–{ALPHA_Q_HI}')
print(f'  Pool T alpha             : {ALPHA_T_LO}–{ALPHA_T_HI} (p={POOL_T_PROB})')


Configuration:
  Source tensors           : /content/drive/MyDrive/SKANN_SSL/ssl_data_30s/tensors/vessel
  Output tensors           : /content/drive/MyDrive/SKANN_SSL/v5_data/tensors/vessel
  Output manifest          : /content/drive/MyDrive/SKANN_SSL/ssl_data_50w/window_pool_50.csv
  Target windows per event : 50
  Phase shift max          : ±51 samples
  Variance sigma           : 0.1
  Pool Q alpha             : 0.1–0.5
  Pool T alpha             : 0.3–0.8 (p=0.2)


## Cell 3 — Load Inputs

In [ ]:
# ── File guards ────────────────────────────────────────────────────────────
if not os.path.exists(VESSEL_FEATURES_PATH):
    raise FileNotFoundError(f'Missing: {VESSEL_FEATURES_PATH}')

# ── Load vessel window features ────────────────────────────────────────────
vdf = pd.read_csv(VESSEL_FEATURES_PATH)
print(f'vessel_window_features.csv : {len(vdf):,} rows, {vdf.event_id.nunique()} events')

# ── Load vessel manifest (event-level provenance) ─────────────────────────
if not os.path.exists(VESSEL_MANIFEST_PATH):
    raise FileNotFoundError(f'Missing: {VESSEL_MANIFEST_PATH}')
vman = pd.read_csv(VESSEL_MANIFEST_PATH)
# Parse timestamps
vman['start'] = pd.to_datetime(vman['start'])
vman['end']   = pd.to_datetime(vman['end'])
# Extract session number from filename (e.g. 'session_042_...' → 42)
vman['session'] = vman['filename'].str.extract(r'(\d+)').astype(int)
print(f'vessel_manifest.csv        : {len(vman):,} events')
print(f'  Columns: {list(vman.columns)}')

# ── Derive split from tensor filename suffix ───────────────────────────────
def derive_split(event_df):
    """Return split from the first tensor filename in the event group."""
    fname = str(event_df.iloc[0]['tensor_file'])
    if '_val.' in fname or fname.endswith('_val.npy'):
        return 'val'
    return 'train'

# ── Load ambient pools — resolve to full .npy paths ───────────────────────
# pool_q_manifest / pool_t_manifest contain tensor_file basenames.
# Resolve against the ambient tensor directories so np.load() works.
pool_q_files, pool_t_files = [], []

for manifest_path, tensor_dir, pool_list, label in [
        (POOL_Q_MANIFEST_PATH, AMBIENT_Q_TENSOR_DIR, pool_q_files, 'Pool Q'),
        (POOL_T_MANIFEST_PATH, AMBIENT_T_TENSOR_DIR, pool_t_files, 'Pool T')]:
    if os.path.exists(manifest_path):
        pm  = pd.read_csv(manifest_path)
        # Find the column that holds the filename
        col = next((c for c in pm.columns
                    if any(k in c.lower() for k in ['tensor', 'file', 'path'])), None)
        fnames = pm[col].tolist() if col else pm.iloc[:, 0].tolist()
        # Resolve to full paths and keep only those that exist on disk
        resolved = []
        for fn in fnames:
            fp = fn if os.path.isabs(fn) else os.path.join(tensor_dir, os.path.basename(fn))
            if os.path.exists(fp):
                resolved.append(fp)
        pool_list.extend(resolved)
        print(f'{label}: {len(fnames):,} in manifest, {len(resolved):,} resolved on disk')
    else:
        print(f'{label} manifest not found — ambient injection disabled for this pool')

print(f'\nEvent window counts:')
wpev = vdf.groupby('event_id').size()
print(f'  min={wpev.min()}  median={wpev.median():.0f}  max={wpev.max()}')
print(f'  Events < {TARGET_WINDOWS} windows : {(wpev < TARGET_WINDOWS).sum()} (will be augmented up)')
print(f'  Events = {TARGET_WINDOWS} windows : {(wpev == TARGET_WINDOWS).sum()} (copied as-is)')
print(f'  Events > {TARGET_WINDOWS} windows : {(wpev > TARGET_WINDOWS).sum()} (diverse subset selected)')


vessel_window_features.csv : 5,846 rows, 141 events
vessel_manifest.csv        : 141 events
  Columns: ['event_id', 'filename', 'split', 'assessment', 'start', 'end', 'dur_sec', 'n_windows', 'can_pair', 'clip_rms', 'mean_tonals', 'blade_rate_hz', 'demon_confidence', 'n_samples', 'session']
Pool Q manifest not found — ambient injection disabled for this pool
Pool T manifest not found — ambient injection disabled for this pool

Event window counts:
  min=16  median=29  max=342
  Events < 50 windows : 107 (will be augmented up)
  Events = 50 windows : 0 (copied as-is)
  Events > 50 windows : 34 (diverse subset selected)


## Cell 4 — Helper Functions

In [ ]:
def iqr_scale(X):
    """Robust IQR scaling. Returns scaled array and (median, iqr) params."""
    med  = np.median(X, axis=0)
    q25  = np.percentile(X, 25, axis=0)
    q75  = np.percentile(X, 75, axis=0)
    iqr  = np.where((q75 - q25) < 1e-8, 1e-8, q75 - q25)
    return (X - med) / iqr


def greedy_diverse_subset(X_scaled, n):
    """
    Greedy max-coverage selection of n rows from X_scaled.
    Seed: row furthest from centroid.
    Each step: add row maximising minimum cosine distance to selected set.
    Returns indices of selected rows.
    """
    if len(X_scaled) <= n:
        return list(range(len(X_scaled)))

    centroid = X_scaled.mean(axis=0)
    norms    = np.linalg.norm(X_scaled, axis=1, keepdims=True)
    norms    = np.where(norms < 1e-12, 1e-12, norms)
    X_norm   = X_scaled / norms

    c_norm   = centroid / max(np.linalg.norm(centroid), 1e-12)
    seed     = int(np.argmin(X_norm @ c_norm))   # furthest from centroid
    selected = [seed]
    min_dists = 1.0 - (X_norm @ X_norm[seed])    # cosine dist to seed

    while len(selected) < n:
        # Exclude already selected
        min_dists[selected] = -np.inf
        next_idx = int(np.argmax(min_dists))
        selected.append(next_idx)
        # Update min distances
        new_dists = 1.0 - (X_norm @ X_norm[next_idx])
        min_dists = np.minimum(min_dists, new_dists)

    return selected


def augment_window(x, rng):
    """
    Apply augmentation pipeline to a 1-D float32 tensor.
    1. Phase shift   : np.roll(x, U(-PHASE_SHIFT_MAX, PHASE_SHIFT_MAX))
    2. Variance scale: x * (1 + N(0, VAR_SIGMA))
    3. Ambient inject: x + alpha * ambient_sample (Pool Q or T)
    4. Re-normalise  : z-score
    """
    x = x.copy().astype(np.float32)

    # 1. Phase shift
    shift = rng.integers(-PHASE_SHIFT_MAX, PHASE_SHIFT_MAX + 1)
    x     = np.roll(x, int(shift))

    # 2. Variance scaling
    x = x * float(1.0 + rng.normal(0.0, VAR_SIGMA))

    # 3. Ambient injection
    use_t = (len(pool_t_files) > 0) and (rng.random() < POOL_T_PROB)
    pool  = pool_t_files if use_t else pool_q_files
    if pool:
        alpha_lo = ALPHA_T_LO if use_t else ALPHA_Q_LO
        alpha_hi = ALPHA_T_HI if use_t else ALPHA_Q_HI
        try:
            amb = np.load(pool[rng.integers(0, len(pool))]).astype(np.float32)
            if amb.shape == x.shape:
                x = x + float(rng.uniform(alpha_lo, alpha_hi)) * amb
        except Exception:
            pass   # load failure — skip injection silently

    # 4. Re-normalise
    std = x.std()
    if std > 1e-8:
        x = (x - x.mean()) / std
    return x


print('Helper functions defined.')
print(f'  greedy_diverse_subset — cosine distance greedy max-coverage')
print(f'  augment_window        — phase shift ±{PHASE_SHIFT_MAX}, var σ={VAR_SIGMA}, ambient injection')

Helper functions defined.
  greedy_diverse_subset — cosine distance greedy max-coverage
  augment_window        — phase shift ±51, var σ=0.1, ambient injection


## Cell 5 — Build Normalised 50-Window Pool

In [ ]:
rng           = np.random.default_rng(seed=RANDOM_SEED)
manifest_rows = []
stats         = {'short': 0, 'exact': 0, 'long': 0,
                 'aug_generated': 0, 'copy_errors': 0}

events = sorted(vdf['event_id'].unique())
print(f'Processing {len(events)} events → {len(events) * TARGET_WINDOWS} total windows')
print(f'Output tensor dir: {OUT_TENSOR_DIR}\n')

for i, eid in enumerate(events):
    ev     = vdf[vdf['event_id'] == eid].reset_index(drop=True)
    n_orig = len(ev)
    split  = derive_split(ev)

    # ── IQR-scaled features for diversity selection ────────────────────────
    X_raw    = ev[FEAT_COLS].values.astype(float)
    X_scaled = iqr_scale(X_raw) if n_orig > 1 else X_raw

    # ── Select original windows ────────────────────────────────────────────
    if n_orig >= TARGET_WINDOWS:
        sel_idx    = greedy_diverse_subset(X_scaled, TARGET_WINDOWS)
        ev_sel     = ev.iloc[sel_idx].reset_index(drop=True)
        aug_needed = 0
        stats['long'] += 1
    else:
        ev_sel     = ev.copy()
        aug_needed = TARGET_WINDOWS - n_orig
        if n_orig == TARGET_WINDOWS:
            stats['exact'] += 1
        else:
            stats['short'] += 1

    # ── Copy original windows to OUT_TENSOR_DIR ────────────────────────────
    for _, row in ev_sel.iterrows():
        src_path = row['tensor_file']
        if not os.path.isabs(src_path):
            src_path = os.path.join(TENSOR_BASE_DIR, src_path)

        out_fname = os.path.basename(row['tensor_file'])
        out_path  = os.path.join(OUT_TENSOR_DIR, out_fname)

        try:
            if not os.path.exists(out_path):          # skip if already copied
                x = np.load(src_path).astype(np.float32)
                np.save(out_path, x)
        except Exception as e:
            stats['copy_errors'] += 1
            print(f'  COPY ERROR {src_path}: {e}')
            continue

        manifest_rows.append({
            'tensor_file' : out_fname,
            'event_id'    : eid,
            'offset_sec'  : row['offset_sec'],
            'source'      : 'original',
            'split'       : split,
            'pre_norm_rms': row['pre_norm_rms'],
            **{c: row[c] for c in FEAT_COLS},
        })

    # ── Generate augmented windows ─────────────────────────────────────────
    aug_counter  = 0
    pool_indices = list(range(len(ev)))   # sample from ALL originals

    while aug_counter < aug_needed:
        src_row  = ev.iloc[rng.choice(pool_indices)]
        src_path = src_row['tensor_file']
        if not os.path.isabs(src_path):
            src_path = os.path.join(TENSOR_BASE_DIR, src_path)

        try:
            x_orig = np.load(src_path).astype(np.float32)
        except Exception as e:
            stats['copy_errors'] += 1
            aug_counter += 1
            print(f'  AUG LOAD ERROR {src_path}: {e}')
            continue

        x_aug    = augment_window(x_orig, rng)
        # Unique filename: event_id + offset + aug index — no collision possible
        aug_fname = f'v_{eid}_{int(src_row["offset_sec"]):07d}_aug{aug_counter:03d}_{split}.npy'
        aug_path  = os.path.join(OUT_TENSOR_DIR, aug_fname)
        np.save(aug_path, x_aug)

        manifest_rows.append({
            'tensor_file' : aug_fname,
            'event_id'    : eid,
            'offset_sec'  : float(src_row['offset_sec']),
            'source'      : 'augmented',
            'split'       : split,
            'pre_norm_rms': float(x_aug.std()),
            **{c: 0.0 for c in FEAT_COLS},
        })

        aug_counter            += 1
        stats['aug_generated'] += 1

    if (i + 1) % 20 == 0 or (i + 1) == len(events):
        print(f'  [{i+1:>3}/{len(events)}] event {eid}  '
              f'orig={n_orig}  aug={aug_needed}  split={split}')

print(f'\n=== NORMALISATION COMPLETE ===')
print(f'  Short events (augmented up) : {stats["short"]}')
print(f'  Exact events (copied as-is) : {stats["exact"]}')
print(f'  Long events (subsampled)    : {stats["long"]}')
print(f'  Augmented windows generated : {stats["aug_generated"]}')
print(f'  Copy/load errors            : {stats["copy_errors"]}')
print(f'  Total manifest rows         : {len(manifest_rows)}')
expected = len(events) * TARGET_WINDOWS
print(f'  Expected                    : {expected}')
if len(manifest_rows) != expected:
    print(f'  WARNING: {expected - len(manifest_rows)} rows missing — check errors above')


Processing 141 events → 7050 total windows
Output tensor dir: /content/drive/MyDrive/SKANN_SSL/v5_data/tensors/vessel

  [ 20/141] event 46  orig=95  aug=0  split=train
  [ 40/141] event 89  orig=25  aug=25  split=train
  [ 60/141] event 117  orig=59  aug=0  split=train
  [ 80/141] event 144  orig=24  aug=26  split=train
  [100/141] event 181  orig=40  aug=10  split=train
  [120/141] event 214  orig=40  aug=10  split=val
  [140/141] event 251  orig=42  aug=8  split=train
  [141/141] event 252  orig=27  aug=23  split=train

=== NORMALISATION COMPLETE ===
  Short events (augmented up) : 107
  Exact events (copied as-is) : 0
  Long events (subsampled)    : 34
  Augmented windows generated : 2656
  Copy/load errors            : 0
  Total manifest rows         : 7050
  Expected                    : 7050


## Cell 6 — Save Manifest + Validate

In [ ]:
pool = pd.DataFrame(manifest_rows)

# ── Integrity checks ───────────────────────────────────────────────────────
assert pool.isnull().sum().sum() == 0, \
    f'Nulls found:\n{pool.isnull().sum()}'
print('Null check                   : PASSED')

counts = pool.groupby('event_id').size()
assert (counts == TARGET_WINDOWS).all(), \
    f'Not all events have {TARGET_WINDOWS} windows:\n{counts[counts != TARGET_WINDOWS]}'
print(f'All events = {TARGET_WINDOWS} windows    : PASSED')

dupes = pool['tensor_file'].duplicated().sum()
assert dupes == 0, f'Duplicate tensor_file entries: {dupes}'
print('No duplicate filenames       : PASSED')

# ── Verify all tensor files exist on disk ──────────────────────────────────
missing_files = [f for f in pool['tensor_file']
                 if not os.path.exists(os.path.join(OUT_TENSOR_DIR, f))]
if missing_files:
    print(f'WARNING: {len(missing_files)} tensor files missing from {OUT_TENSOR_DIR}')
    for f in missing_files[:5]:
        print(f'  {f}')
else:
    print(f'All {len(pool):,} tensor files present on disk : PASSED')

# ── Statistics ─────────────────────────────────────────────────────────────
print('\n=== POOL STATISTICS ===')
print(f'  Total windows : {len(pool):,}  ({pool.event_id.nunique()} events × {TARGET_WINDOWS})')
print(f'  Tensor dir    : {OUT_TENSOR_DIR}')
print(f'\n  Source distribution:')
print(pool['source'].value_counts().to_string())
print(f'\n  Split distribution:')
print(pool['split'].value_counts().to_string())
print(f'\n  Split × source:')
print(pool.groupby(["split","source"]).size().unstack(fill_value=0).to_string())

# ── Join event-level provenance from vessel_manifest ──────────────────────
# Columns joined: session, recording_file, event_start, event_end,
#                 dur_sec, assessment, blade_rate_hz, demon_confidence
prov = vman[['event_id','session','filename','start','end',
             'dur_sec','assessment','blade_rate_hz','demon_confidence']].copy()
prov = prov.rename(columns={
    'filename'        : 'recording_file',
    'start'           : 'event_start',
    'end'             : 'event_end',
    'dur_sec'         : 'event_dur_sec',
})
pool = pool.merge(prov, on='event_id', how='left')
unmatched = pool['recording_file'].isna().sum()
if unmatched:
    print(f'WARNING: {unmatched} windows could not be matched to vessel_manifest')
else:
    print(f'Provenance join              : PASSED ({len(pool):,} windows matched)')

# ── Compute window-level time bounds from offset_sec ──────────────────────
# Each window is 30s (15360 samples @ 512 Hz). offset_sec is start of window
# relative to start of recording session, not event start.
WINDOW_SEC = 15360 / 512  # = 30.0s
pool['window_start'] = pool['event_start'] + pd.to_timedelta(pool['offset_sec'], unit='s')
pool['window_end']   = pool['window_start'] + pd.to_timedelta(WINDOW_SEC, unit='s')
# Augmented windows inherit their source window's time (marked in source column)

# ── Save manifest ──────────────────────────────────────────────────────────
col_order = [
    'tensor_file', 'event_id', 'session', 'recording_file',
    'event_start', 'event_end', 'event_dur_sec',
    'window_start', 'window_end', 'offset_sec',
    'source', 'split', 'assessment', 'blade_rate_hz', 'demon_confidence',
    'pre_norm_rms',
] + FEAT_COLS
pool[col_order].to_csv(OUT_MANIFEST_PATH, index=False)
sz = os.path.getsize(OUT_MANIFEST_PATH) / 1024
print(f'\nSaved manifest : {OUT_MANIFEST_PATH}')
print(f'Rows  : {len(pool):,}  |  Size: {sz:.1f} KB')

print('\n=== NEXT STEP ===')
print('Run pairing_manifest_generator with:')
print(f'  VESSEL_FEATURES_PATH = "{OUT_MANIFEST_PATH}"')
print(f'  TENSOR_BASE_DIR      = "{OUT_TENSOR_DIR}"')
print(f'  (pairing_manifest.csv output → ssl_data_50w/ or v5_manifests/)')


Null check                   : PASSED
All events = 50 windows    : PASSED
No duplicate filenames       : PASSED
All 7,050 tensor files present on disk : PASSED

=== POOL STATISTICS ===
  Total windows : 7,050  (141 events × 50)
  Tensor dir    : /content/drive/MyDrive/SKANN_SSL/v5_data/tensors/vessel

  Source distribution:
source
original     4394
augmented    2656

  Split distribution:
split
train    5800
val      1250

  Split × source:
source  augmented  original
split                      
train        2175      3625
val           481       769
Provenance join              : PASSED (7,050 windows matched)

Saved manifest : /content/drive/MyDrive/SKANN_SSL/ssl_data_50w/window_pool_50.csv
Rows  : 7,050  |  Size: 2174.8 KB

=== NEXT STEP ===
Run pairing_manifest_generator with:
  VESSEL_FEATURES_PATH = "/content/drive/MyDrive/SKANN_SSL/ssl_data_50w/window_pool_50.csv"
  TENSOR_BASE_DIR      = "/content/drive/MyDrive/SKANN_SSL/v5_data/tensors/vessel"
  (pairing_manifest.csv output → 